## ModelMonitor Usage Example
This notebook demonstrates how to use momo_ml.monitor.ModelMonitor to monitor model drift in production. We will use simulated reference data (e.g., training data) and current data (e.g., a new batch of production data), and observe the results of three types of drift detection.

In [1]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')  # ignore some potential warnings

from momo_ml.monitor.model_monitor import ModelMonitor

# Set random seed for reproducibility
np.random.seed(42)

# Generate reference dataset (1000 samples)
n_ref = 1000
ref_df = pd.DataFrame({
    'age': np.random.normal(35, 10, n_ref).clip(18, 80).astype(int),
    'income': np.random.lognormal(mean=10, sigma=0.5, size=n_ref).astype(int),
    'gender': np.random.choice(['M', 'F'], size=n_ref, p=[0.5, 0.5]),
    'label': np.random.choice([0, 1], size=n_ref, p=[0.7, 0.3]),
    'prediction': np.random.uniform(0, 1, n_ref)
})

# Generate current dataset (1200 samples) with various drifts
n_cur = 1200
# age distribution shifts right (older)
age_cur = np.random.normal(45, 12, n_cur).clip(18, 80).astype(int)
# income distribution shifts upward overall
income_cur = np.random.lognormal(mean=10.5, sigma=0.6, size=n_cur).astype(int)
# gender ratio changes
gender_cur = np.random.choice(['M', 'F'], size=n_cur, p=[0.6, 0.4])
# label distribution changes (more positives)
label_cur = np.random.choice([0, 1], size=n_cur, p=[0.5, 0.5])
# prediction distribution changes (leans toward lower probabilities)
pred_cur = np.random.beta(2, 5, n_cur)

cur_df = pd.DataFrame({
    'age': age_cur,
    'income': income_cur,
    'gender': gender_cur,
    'label': label_cur,
    'prediction': pred_cur
})

print("Reference dataset shape:", ref_df.shape)
print("Current dataset shape:", cur_df.shape)
ref_df.head()

Reference dataset shape: (1000, 5)
Current dataset shape: (1200, 5)


,age,income,gender,label,prediction
0,39,44341,M,0,0.668877
1,33,34972,M,0,0.798656
2,41,22693,M,1,0.932753
3,50,15939,M,0,0.020138
4,32,31229,F,0,0.153784


## Create a ModelMonitor Instance

In [2]:
monitor = ModelMonitor(
    ref_df=ref_df,
    cur_df=cur_df,
    label_col='label',
    pred_col='prediction',
    # Optional: customize each detector's parameters
    data_drift_kwargs={'features': ['age', 'income', 'gender']},  # monitor only selected features
    performance_kwargs={'task_type': 'classification'},           # explicitly specify the task type
    prediction_drift_kwargs={'include_ks': True, 'bins': 15}      # include KS test in prediction drift
)

print("Monitor created, but detectors are not yet instantiated (lazy initialization).")

Monitor created, but detectors are not yet instantiated (lazy initialization).


## Run All Drift Detections

In [3]:
results = monitor.run_all()

# View the top-level keys of the result
print(results.keys())

dict_keys(['performance_drift', 'data_drift', 'prediction_drift'])


## Analyze Performance Drift

In [4]:
perf_result = results['performance_drift']
print("Task type:", perf_result['task_type'])
print("\nReference performance:")
for k, v in perf_result['reference'].items():
    print(f"  {k}: {v:.4f}")
print("\nCurrent performance:")
for k, v in perf_result['current'].items():
    print(f"  {k}: {v:.4f}")
print("\nDelta (current - reference):")
for k, v in perf_result['delta'].items():
    print(f"  {k}: {v:+.4f}")

Task type: classification

Reference performance:
  auc: 0.4648
  ks: 0.0860
  accuracy: 0.4800
  precision: 0.2336
  recall: 0.4385
  f1: 0.3048

Current performance:
  auc: 0.4804
  ks: 0.0570
  accuracy: 0.4892
  precision: 0.5038
  recall: 0.1075
  f1: 0.1772

Delta (current - reference):
  f1: -0.1276
  auc: +0.0156
  precision: +0.2702
  accuracy: +0.0092
  recall: -0.3310
  ks: -0.0289


## Analyze Data Drift

In [6]:
data_result = results['data_drift']
print("Numeric feature drift:")
for feat, metrics in data_result['numeric_features'].items():
    print(f"\n{feat}:")
    for metric_name, value in metrics.items():
        if metric_name == 'ks':
            print(f"  KS statistic: {value['statistic']:.4f}, p-value: {value['pvalue']:.4f}")
        else:
            print(f"  {metric_name}: {value:.4f}")

print("\nCategorical feature drift:")
for feat, metrics in data_result['categorical_features'].items():
    print(f"\n{feat}:")
    for metric_name, value in metrics.items():
        print(f"  {metric_name}: {value:.4f}")

Numeric feature drift:

age:
  psi: 0.7179
  kl: 0.3517
  KS statistic: 0.3653, p-value: 0.0000
  js: 0.0877
  wd: 9.4967

income:
  psi: 0.5527
  kl: 0.2602
  KS statistic: 0.3358, p-value: 0.0000
  js: 0.0670
  wd: 16973.9450

Categorical feature drift:

gender:
  psi: 0.0733
  kl: 0.0371
  js: 0.0091
  wd: 0.1338


## Analyze Prediction Drift

In [7]:
pred_result = results['prediction_drift']
print("Prediction type:", pred_result['prediction_type'])

print("\nSummary statistics:")
if pred_result['prediction_type'] == 'continuous':
    stats = pred_result['summary_statistics']
    for stat, val in stats.items():
        if isinstance(val, dict):
            print(f"  {stat}: reference={val['reference']:.4f}, current={val['current']:.4f}, delta={val.get('delta', 'N/A')}")
        else:
            print(f"  {stat}: {val}")
else:
    # categorical case
    stats = pred_result['summary_statistics']
    print("  Category proportion changes:")
    for cat, delta in stats['delta_proportions'].items():
        print(f"    {cat}: reference={stats['reference_proportions'][cat]:.4f}, "
              f"current={stats['current_proportions'][cat]:.4f}, delta={delta:+.4f}")

print("\nDistribution shift metrics:")
for metric, value in pred_result['distribution_shift'].items():
    if metric == 'ks':
        print(f"  KS: statistic={value['statistic']:.4f}, p-value={value['pvalue']:.4f}")
    else:
        print(f"  {metric}: {value:.4f}")

if 'decile_shift' in pred_result:
    print("\nDecile shift:")
    decile = pred_result['decile_shift']
    for i, q in enumerate(decile['quantiles']):
        print(f"  {q*100:.0f}%: reference={decile['ref_values'][i]:.4f}, "
              f"current={decile['cur_values'][i]:.4f}, delta={decile['delta'][i]:+.4f}")

Prediction type: continuous

Summary statistics:
  mean: reference=0.4987, current=0.2835, delta=-0.21519761334903587
  std: reference=0.2880, current=0.1558, delta=-0.13223281498630218
  min: reference=0.0002, current=0.0082, delta=N/A
  max: reference=0.9988, current=0.8063, delta=N/A
  q25: reference=0.2523, current=0.1647, delta=N/A
  q50: reference=0.4837, current=0.2588, delta=N/A
  q75: reference=0.7518, current=0.3830, delta=N/A

Distribution shift metrics:
  l1_distance: 12.0823
  l2_distance: 3.5321
  psi: 3.0305
  KS: statistic=0.3917, p-value=0.0000

Decile shift:
  0%: reference=0.0002, current=0.0082, delta=+0.0080
  10%: reference=0.1068, current=0.0968, delta=-0.0100
  20%: reference=0.2041, current=0.1409, delta=-0.0632
  30%: reference=0.3068, current=0.1849, delta=-0.1220
  40%: reference=0.3877, current=0.2199, delta=-0.1677
  50%: reference=0.4837, current=0.2588, delta=-0.2249
  60%: reference=0.5832, current=0.3030, delta=-0.2801
  70%: reference=0.7009, current=

## Run Specific Detections Individually

In [8]:
# Run only data drift
data_only = monitor.run_data_drift()
print("Numeric features in data drift result:", list(data_only['numeric_features'].keys()))

Numeric features in data drift result: ['age', 'income']


## Access the Underlying Detectors

In [9]:
# Access the DataDriftDetector instance
dd = monitor.data_drift
print("Feature list:", dd.features)

# Compute PSI for a single feature
psi_age = dd.compute_feature_psi('age')
print("PSI for age:", psi_age)

Feature list: ['age', 'income', 'gender']
PSI for age: 0.7179435460952837


## Run Again with Customized Parameters

In [10]:
# Create a monitor that only monitors age and income, using a different number of bins for KL
monitor2 = ModelMonitor(
    ref_df, cur_df,
    data_drift_kwargs={'features': ['age', 'income'], 'kl_buckets': 20}
)
data2 = monitor2.run_data_drift()
print("Feature list (monitor2):", monitor2.data_drift.features)
print("KL divergence for age:", data2['numeric_features']['age']['kl'])

Feature list (monitor2): ['age', 'income']
KL divergence for age: 0.3781040271943525


## Summary
With ModelMonitor, we can easily combine data drift, performance drift, and prediction drift detection, and quickly assess the health of a model in production. In practice, you could run this monitor periodically and trigger alerts or model retraining based on the drift indicators.

This example covered basic usage; more advanced configurations (e.g., custom quantiles, selecting specific drift metrics) are possible by passing the appropriate kwargs.